# 🚀 GPU-Accelerated High-Volume Sepsis Extraction (FIXED)

## Target: 50,000+ Samples

**Run cells in order!**

In [ ]:
# ============================================================================
# CELL 0: SETUP - Install & Authenticate
# ============================================================================
# RUN THIS FIRST!
# ============================================================================

# Install required packages
!pip install -q google-cloud-bigquery db-dtypes pyarrow tqdm joblib

# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

print("✅ Setup complete! Proceed to Cell 1.")

In [ ]:
# ============================================================================
# CELL 1: CONFIGURATION (FIXED - Auto-detects Project ID)
# ============================================================================

import numpy as np
import pandas as pd
from google.cloud import bigquery
import torch
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import cpu_count
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings
import subprocess
warnings.filterwarnings('ignore')

# ============================================================================
# AUTO-DETECT PROJECT ID (FIXED!)
# ============================================================================

def get_project_id():
    """Auto-detect Google Cloud project ID."""
    # Method 1: gcloud config
    try:
        result = subprocess.run(
            ['gcloud', 'config', 'get-value', 'project'],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            project = result.stdout.strip()
            if project and project != '(unset)':
                return project
    except:
        pass
    
    # Method 2: BigQuery default client
    try:
        temp_client = bigquery.Client()
        if temp_client.project:
            return temp_client.project
    except:
        pass
    
    return None

# Auto-detect
PROJECT_ID = get_project_id()

if PROJECT_ID:
    print(f"✅ Auto-detected project: {PROJECT_ID}")
else:
    print("⚠️ Could not auto-detect project ID.")
    PROJECT_ID = input("Enter your Google Cloud Project ID: ").strip()

if not PROJECT_ID:
    raise ValueError("❌ Project ID is required!")

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'target_samples': 50000,
    'control_ratio': 5,
    'min_icu_hours': 12,
    'min_data_hours': 4,
    'prediction_gap_hours': 2,
    'sofa_increase_threshold': 2,
    'observation_window_hours': 24,
    'chunk_size': 5000,
    'use_gpu': torch.cuda.is_available(),
    'project_id': PROJECT_ID,
}

DEVICE = torch.device('cuda' if CONFIG['use_gpu'] else 'cpu')
N_CORES = cpu_count()

print(f"\n{'='*60}")
print("CONFIGURATION")
print(f"{'='*60}")
print(f"  Project ID: {CONFIG['project_id']}")
print(f"  Target: {CONFIG['target_samples']:,} samples")
print(f"  Control ratio: 1:{CONFIG['control_ratio']}")
print(f"  GPU: {DEVICE}")
print(f"  CPU Cores: {N_CORES}")
print(f"{'='*60}")

# Initialize BigQuery client
client = bigquery.Client(project=CONFIG['project_id'])

# Test connection
try:
    test = client.query("SELECT 1 as test").to_dataframe()
    print("\n✅ BigQuery connection successful!")
except Exception as e:
    print(f"\n❌ Connection failed: {e}")
    raise

In [ ]:
# ============================================================================
# CELL 2: GET ICU STAYS
# ============================================================================

print("="*70)
print("HIGH-VOLUME DATA EXTRACTION")
print("="*70)

print(f"\n[1/8] Getting ICU stays (min {CONFIG['min_icu_hours']}h)...")

query_stays = f"""
SELECT 
    icu.stay_id,
    icu.subject_id,
    icu.hadm_id,
    icu.intime,
    icu.outtime,
    TIMESTAMP_DIFF(icu.outtime, icu.intime, HOUR) as los_hours,
    pat.gender,
    pat.anchor_age as age,
    ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) as stay_num
FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat
    ON icu.subject_id = pat.subject_id
WHERE TIMESTAMP_DIFF(icu.outtime, icu.intime, HOUR) >= {CONFIG['min_icu_hours']}
ORDER BY icu.subject_id, icu.intime
"""

stays_df = client.query(query_stays).to_dataframe()
print(f"   ✅ Total ICU stays: {len(stays_df):,}")
print(f"   ✅ Unique patients: {stays_df['subject_id'].nunique():,}")

In [ ]:
# ============================================================================
# CELL 3: GET SUSPECTED INFECTIONS
# ============================================================================

print(f"\n[2/8] Identifying suspected infections...")

query_infection = """
WITH antibiotics AS (
    SELECT DISTINCT
        ie.stay_id,
        ie.subject_id,
        MIN(ie.starttime) as abx_time
    FROM `physionet-data.mimiciv_3_1_icu.inputevents` ie
    WHERE ie.itemid IN (
        225798, 225842, 225843, 225844, 225845, 225846, 225847,
        225848, 225849, 225850, 225851, 225853, 225855, 225857,
        225859, 225860, 225862, 225863, 225865, 225866, 225868,
        225869, 225871, 225873, 225875, 225876, 225877, 225879,
        225881, 225882, 225883, 225884, 225885, 225886, 225888,
        225889, 225890, 225892, 225893, 225895, 225896, 225897,
        225898, 225899, 227689
    )
    GROUP BY ie.stay_id, ie.subject_id
),
cultures AS (
    SELECT DISTINCT
        icu.stay_id,
        icu.subject_id,
        MIN(me.charttime) as culture_time
    FROM `physionet-data.mimiciv_3_1_hosp.microbiologyevents` me
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu
        ON me.subject_id = icu.subject_id
        AND me.charttime BETWEEN icu.intime AND icu.outtime
    GROUP BY icu.stay_id, icu.subject_id
)
SELECT 
    COALESCE(a.stay_id, c.stay_id) as stay_id,
    COALESCE(a.subject_id, c.subject_id) as subject_id,
    a.abx_time,
    c.culture_time,
    LEAST(
        COALESCE(a.abx_time, c.culture_time),
        COALESCE(c.culture_time, a.abx_time)
    ) as suspected_infection_time
FROM antibiotics a
FULL OUTER JOIN cultures c ON a.stay_id = c.stay_id
WHERE a.stay_id IS NOT NULL OR c.stay_id IS NOT NULL
"""

infection_df = client.query(query_infection).to_dataframe()
print(f"   ✅ Stays with suspected infection: {len(infection_df):,}")

In [ ]:
# ============================================================================
# CELL 4: GET SOFA COMPONENT DATA (Chunked for efficiency)
# ============================================================================

print(f"\n[3/8] Extracting SOFA components (chunked)...")

infected_stays = infection_df['stay_id'].dropna().astype(int).tolist()
print(f"   Processing {len(infected_stays):,} infected stays...")

def get_sofa_chunk(stay_ids_chunk):
    """Get SOFA data for a chunk of stay IDs."""
    stays_str = ','.join(map(str, stay_ids_chunk))
    
    query = f"""
    -- Respiratory: PaO2
    SELECT stay_id, charttime, 'pao2' as component, valuenum as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid IN (220224, 490) AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    
    UNION ALL
    
    -- Respiratory: FiO2
    SELECT stay_id, charttime, 'fio2' as component, 
           CASE WHEN valuenum > 1 THEN valuenum/100.0 ELSE valuenum END as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid IN (223835, 190) AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    
    UNION ALL
    
    -- Cardiovascular: MAP
    SELECT stay_id, charttime, 'map' as component, valuenum as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid = 220052 AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    
    UNION ALL
    
    -- Neurological: GCS components
    SELECT stay_id, charttime, 'gcs' as component, valuenum as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid IN (223900, 223901, 220739) AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    
    UNION ALL
    
    -- Coagulation: Platelets
    SELECT icu.stay_id, le.charttime, 'platelets' as component, le.valuenum as value
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu 
        ON le.subject_id = icu.subject_id
        AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE le.itemid = 51265 AND le.valuenum IS NOT NULL AND icu.stay_id IN ({stays_str})
    
    UNION ALL
    
    -- Liver: Bilirubin
    SELECT icu.stay_id, le.charttime, 'bilirubin' as component, le.valuenum as value
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu 
        ON le.subject_id = icu.subject_id
        AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE le.itemid = 50885 AND le.valuenum IS NOT NULL AND icu.stay_id IN ({stays_str})
    
    UNION ALL
    
    -- Renal: Creatinine
    SELECT icu.stay_id, le.charttime, 'creatinine' as component, le.valuenum as value
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu 
        ON le.subject_id = icu.subject_id
        AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE le.itemid = 50912 AND le.valuenum IS NOT NULL AND icu.stay_id IN ({stays_str})
    """
    return client.query(query).to_dataframe()

# Process in chunks
chunk_size = 8000
sofa_dfs = []

for i in tqdm(range(0, len(infected_stays), chunk_size), desc="   SOFA chunks"):
    chunk = infected_stays[i:i+chunk_size]
    try:
        df = get_sofa_chunk(chunk)
        if len(df) > 0:
            sofa_dfs.append(df)
    except Exception as e:
        print(f"   ⚠️ Chunk {i//chunk_size} warning: {str(e)[:50]}")
        continue

sofa_data = pd.concat(sofa_dfs, ignore_index=True) if sofa_dfs else pd.DataFrame()
print(f"\n   ✅ SOFA measurements: {len(sofa_data):,}")
print(f"   ✅ Stays with SOFA data: {sofa_data['stay_id'].nunique():,}")

In [ ]:
# ============================================================================
# CELL 5: CALCULATE SOFA SCORES (GPU-accelerated)
# ============================================================================

print(f"\n[4/8] Computing SOFA scores...")

def compute_hourly_sofa(stay_id, stay_data, intime):
    """Compute hourly SOFA scores for a single stay."""
    if len(stay_data) == 0:
        return None
    
    stay_data = stay_data.copy()
    stay_data['hours'] = (stay_data['charttime'] - intime).dt.total_seconds() / 3600
    stay_data['hour_bin'] = stay_data['hours'].astype(int)
    
    # Pivot and aggregate
    hourly = stay_data.groupby(['hour_bin', 'component'])['value'].mean().unstack(fill_value=np.nan)
    hourly = hourly.ffill().bfill()
    
    results = []
    for hour in hourly.index:
        row = hourly.loc[hour]
        sofa = 0
        
        # Respiratory (PaO2/FiO2)
        if 'pao2' in row and 'fio2' in row:
            if pd.notna(row['pao2']) and pd.notna(row['fio2']) and row['fio2'] > 0:
                pf = row['pao2'] / row['fio2']
                if pf < 100: sofa += 4
                elif pf < 200: sofa += 3
                elif pf < 300: sofa += 2
                elif pf < 400: sofa += 1
        
        # Coagulation (Platelets)
        if 'platelets' in row and pd.notna(row['platelets']):
            plt = row['platelets']
            if plt < 20: sofa += 4
            elif plt < 50: sofa += 3
            elif plt < 100: sofa += 2
            elif plt < 150: sofa += 1
        
        # Liver (Bilirubin)
        if 'bilirubin' in row and pd.notna(row['bilirubin']):
            bili = row['bilirubin']
            if bili >= 12: sofa += 4
            elif bili >= 6: sofa += 3
            elif bili >= 2: sofa += 2
            elif bili >= 1.2: sofa += 1
        
        # Cardiovascular (MAP)
        if 'map' in row and pd.notna(row['map']):
            if row['map'] < 70: sofa += 1
        
        # Neurological (GCS)
        if 'gcs' in row and pd.notna(row['gcs']):
            gcs = row['gcs']
            if gcs < 6: sofa += 4
            elif gcs < 10: sofa += 3
            elif gcs < 13: sofa += 2
            elif gcs < 15: sofa += 1
        
        # Renal (Creatinine)
        if 'creatinine' in row and pd.notna(row['creatinine']):
            cr = row['creatinine']
            if cr >= 5.0: sofa += 4
            elif cr >= 3.5: sofa += 3
            elif cr >= 2.0: sofa += 2
            elif cr >= 1.2: sofa += 1
        
        results.append({'hour': hour, 'sofa': sofa})
    
    if results:
        df = pd.DataFrame(results)
        df['stay_id'] = stay_id
        return df
    return None

# Create lookup for intime
stays_info = stays_df.set_index('stay_id')['intime'].to_dict()
sofa_grouped = sofa_data.groupby('stay_id')

# Process with parallel execution
def process_stay(stay_id):
    if stay_id not in stays_info:
        return None
    try:
        stay_data = sofa_grouped.get_group(stay_id)
        return compute_hourly_sofa(stay_id, stay_data, stays_info[stay_id])
    except:
        return None

unique_stays = sofa_data['stay_id'].unique()
print(f"   Processing {len(unique_stays):,} stays...")

# Parallel processing
sofa_results = Parallel(n_jobs=N_CORES, backend='threading')(
    delayed(process_stay)(sid) for sid in tqdm(unique_stays, desc="   SOFA calc")
)

sofa_results = [r for r in sofa_results if r is not None]
all_sofa_df = pd.concat(sofa_results, ignore_index=True)

print(f"\n   ✅ SOFA scores computed: {len(all_sofa_df):,}")
print(f"   ✅ Stays with SOFA: {all_sofa_df['stay_id'].nunique():,}")

In [ ]:
# ============================================================================
# CELL 6: IDENTIFY SEPSIS CASES
# ============================================================================

print(f"\n[5/8] Identifying Sepsis-3 cases...")

def identify_sepsis(stay_id, sofa_df, min_data_hours):
    """Identify sepsis onset for a single stay."""
    stay_sofa = sofa_df[sofa_df['stay_id'] == stay_id].sort_values('hour')
    
    if len(stay_sofa) < 2:
        return None
    
    # Baseline SOFA (first 6 hours)
    baseline = stay_sofa[stay_sofa['hour'] <= 6]['sofa'].min()
    if pd.isna(baseline):
        baseline = stay_sofa['sofa'].iloc[0]
    
    # Find first SOFA increase >= 2
    for _, row in stay_sofa.iterrows():
        if row['sofa'] >= baseline + CONFIG['sofa_increase_threshold']:
            onset_hour = row['hour']
            if onset_hour >= min_data_hours + CONFIG['prediction_gap_hours']:
                return {
                    'stay_id': stay_id,
                    'onset_hour': onset_hour,
                    'baseline_sofa': baseline,
                    'onset_sofa': row['sofa']
                }
    return None

# Filter to infected stays
infected_set = set(infection_df['stay_id'].dropna().astype(int))
sofa_stays = all_sofa_df['stay_id'].unique()
eligible_stays = [s for s in sofa_stays if s in infected_set]

print(f"   Checking {len(eligible_stays):,} eligible stays...")

sepsis_results = Parallel(n_jobs=N_CORES, backend='threading')(
    delayed(identify_sepsis)(sid, all_sofa_df, CONFIG['min_data_hours']) 
    for sid in tqdm(eligible_stays, desc="   Sepsis ID")
)

sepsis_cases = [r for r in sepsis_results if r is not None]
sepsis_df = pd.DataFrame(sepsis_cases)

print(f"\n   ✅ Sepsis-3 cases: {len(sepsis_df):,}")
if len(sepsis_df) > 0:
    print(f"   Mean onset hour: {sepsis_df['onset_hour'].mean():.1f}")

In [ ]:
# ============================================================================
# CELL 7: CREATE MATCHED CONTROLS (1:5 ratio)
# ============================================================================

print(f"\n[6/8] Creating matched controls (1:{CONFIG['control_ratio']})...")

# Merge sepsis with stay info
sepsis_full = sepsis_df.merge(
    stays_df[['stay_id', 'subject_id', 'intime', 'age', 'gender', 'los_hours']],
    on='stay_id'
)

# Get control candidates
sepsis_stay_ids = set(sepsis_df['stay_id'])
control_pool = stays_df[
    (~stays_df['stay_id'].isin(sepsis_stay_ids)) &
    (stays_df['los_hours'] >= CONFIG['min_icu_hours'])
].copy()

print(f"   Sepsis cases: {len(sepsis_full):,}")
print(f"   Control pool: {len(control_pool):,}")

# Match controls
used_controls = set()
all_controls = []

for _, sep_row in tqdm(sepsis_full.iterrows(), total=len(sepsis_full), desc="   Matching"):
    # Find matching controls
    candidates = control_pool[
        (control_pool['age'] >= sep_row['age'] - 10) &
        (control_pool['age'] <= sep_row['age'] + 10) &
        (control_pool['los_hours'] >= sep_row['onset_hour'] + 12) &
        (~control_pool['stay_id'].isin(used_controls))
    ]
    
    n_match = min(CONFIG['control_ratio'], len(candidates))
    if n_match > 0:
        sampled = candidates.sample(n_match, random_state=42)
        for _, ctrl in sampled.iterrows():
            all_controls.append({
                'stay_id': ctrl['stay_id'],
                'subject_id': ctrl['subject_id'],
                'intime': ctrl['intime'],
                'age': ctrl['age'],
                'gender': ctrl['gender'],
                'onset_hour': sep_row['onset_hour'],
                'label': 0
            })
            used_controls.add(ctrl['stay_id'])

control_df = pd.DataFrame(all_controls)

# Create final cohort
sepsis_cohort = sepsis_full[['stay_id', 'subject_id', 'intime', 'age', 'gender', 'onset_hour']].copy()
sepsis_cohort['label'] = 1

cohort_df = pd.concat([sepsis_cohort, control_df], ignore_index=True)

print(f"\n   ✅ Final cohort: {len(cohort_df):,}")
print(f"   Sepsis: {(cohort_df['label']==1).sum():,} ({(cohort_df['label']==1).mean()*100:.1f}%)")
print(f"   Control: {(cohort_df['label']==0).sum():,} ({(cohort_df['label']==0).mean()*100:.1f}%)")

In [ ]:
# ============================================================================
# CELL 8: EXTRACT FEATURES
# ============================================================================

print(f"\n[7/8] Extracting features...")

FEATURE_VITALS = {
    220045: 'heart_rate', 220179: 'sbp', 220050: 'sbp',
    220180: 'dbp', 220051: 'dbp', 220052: 'map',
    220210: 'resp_rate', 220277: 'spo2',
    223761: 'temperature', 223762: 'temperature',
    220739: 'gcs_eye', 223900: 'gcs_verbal', 223901: 'gcs_motor',
}

FEATURE_LABS = {
    51301: 'wbc', 50912: 'creatinine', 51265: 'platelets',
    50885: 'bilirubin', 50813: 'lactate', 50931: 'glucose',
    51006: 'bun', 50983: 'sodium', 50971: 'potassium', 51222: 'hemoglobin',
}

ALL_FEATURES = {**FEATURE_VITALS, **FEATURE_LABS}
FEATURE_NAMES = list(set(ALL_FEATURES.values()))

cohort_stay_ids = cohort_df['stay_id'].tolist()

# Extract vitals
print("   Extracting vitals...")
vitals_dfs = []
vital_items = ','.join(map(str, FEATURE_VITALS.keys()))

for i in tqdm(range(0, len(cohort_stay_ids), CONFIG['chunk_size']), desc="   Vitals"):
    chunk = cohort_stay_ids[i:i+CONFIG['chunk_size']]
    stays_str = ','.join(map(str, chunk))
    
    query = f"""
    SELECT stay_id, charttime, itemid, valuenum
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE stay_id IN ({stays_str}) AND itemid IN ({vital_items}) AND valuenum IS NOT NULL
    """
    df = client.query(query).to_dataframe()
    if len(df) > 0:
        vitals_dfs.append(df)

vitals_raw = pd.concat(vitals_dfs, ignore_index=True) if vitals_dfs else pd.DataFrame()
print(f"   Vitals: {len(vitals_raw):,} rows")

# Extract labs
print("   Extracting labs...")
labs_dfs = []
lab_items = ','.join(map(str, FEATURE_LABS.keys()))

for i in tqdm(range(0, len(cohort_stay_ids), CONFIG['chunk_size']), desc="   Labs"):
    chunk = cohort_stay_ids[i:i+CONFIG['chunk_size']]
    stays_str = ','.join(map(str, chunk))
    
    query = f"""
    SELECT icu.stay_id, le.charttime, le.itemid, le.valuenum
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu
        ON le.subject_id = icu.subject_id
        AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE icu.stay_id IN ({stays_str}) AND le.itemid IN ({lab_items}) AND le.valuenum IS NOT NULL
    """
    df = client.query(query).to_dataframe()
    if len(df) > 0:
        labs_dfs.append(df)

labs_raw = pd.concat(labs_dfs, ignore_index=True) if labs_dfs else pd.DataFrame()
print(f"   Labs: {len(labs_raw):,} rows")

# Combine
df_raw = pd.concat([vitals_raw, labs_raw], ignore_index=True)
df_raw['feature'] = df_raw['itemid'].map(ALL_FEATURES)
print(f"\n   ✅ Total measurements: {len(df_raw):,}")

In [ ]:
# ============================================================================
# CELL 9: CREATE FEATURE TENSORS
# ============================================================================

print(f"\n[8/8] Creating feature tensors...")

N_TIMESTEPS = CONFIG['observation_window_hours']
N_FEATURES = len(FEATURE_NAMES)

def safe_divide(a, b, default=0.0):
    """Safe division without warnings."""
    result = np.full_like(a, default, dtype=np.float64)
    mask = (b != 0) & np.isfinite(b) & np.isfinite(a)
    np.divide(a, b, out=result, where=mask)
    return result

def process_stay_tensor(stay_id, stay_data, cohort_row):
    """Create feature tensor for a single stay."""
    intime = cohort_row['intime']
    onset_hour = cohort_row['onset_hour']
    
    end_hour = onset_hour - CONFIG['prediction_gap_hours']
    start_hour = max(0, end_hour - N_TIMESTEPS)
    
    if end_hour <= start_hour:
        return None
    
    stay_data = stay_data.copy()
    stay_data['hours'] = (stay_data['charttime'] - intime).dt.total_seconds() / 3600
    stay_data = stay_data[(stay_data['hours'] >= start_hour) & (stay_data['hours'] < end_hour)]
    
    if len(stay_data) < 5:
        return None
    
    stay_data['time_bin'] = ((stay_data['hours'] - start_hour)).astype(int).clip(0, N_TIMESTEPS-1)
    
    # Initialize tensor
    features = np.zeros((N_TIMESTEPS, N_FEATURES), dtype=np.float32)
    
    for fname in FEATURE_NAMES:
        fdata = stay_data[stay_data['feature'] == fname]
        if len(fdata) > 0:
            fidx = FEATURE_NAMES.index(fname)
            for tb in range(N_TIMESTEPS):
                vals = fdata[fdata['time_bin'] == tb]['valuenum']
                if len(vals) > 0:
                    features[tb, fidx] = vals.mean()
    
    # Forward fill
    features = pd.DataFrame(features).ffill().bfill().fillna(0).values.astype(np.float32)
    
    # Add derived features
    derived = []
    
    # Shock index
    if 'heart_rate' in FEATURE_NAMES and 'sbp' in FEATURE_NAMES:
        hr = features[:, FEATURE_NAMES.index('heart_rate')]
        sbp = features[:, FEATURE_NAMES.index('sbp')]
        derived.append(safe_divide(hr, sbp))
    
    # BUN/Cr ratio
    if 'bun' in FEATURE_NAMES and 'creatinine' in FEATURE_NAMES:
        bun = features[:, FEATURE_NAMES.index('bun')]
        cr = features[:, FEATURE_NAMES.index('creatinine')]
        derived.append(safe_divide(bun, cr))
    
    if derived:
        derived_arr = np.column_stack(derived)
        features = np.concatenate([features, derived_arr], axis=1)
    
    return features

# Process all stays
cohort_lookup = cohort_df.set_index('stay_id').to_dict('index')
df_raw_grouped = df_raw.groupby('stay_id')

X_list, y_list = [], []
stay_ids_processed, subject_ids_processed = [], []

for stay_id in tqdm(cohort_df['stay_id'].unique(), desc="   Tensors"):
    if stay_id not in cohort_lookup:
        continue
    try:
        stay_data = df_raw_grouped.get_group(stay_id)
    except:
        continue
    
    cohort_row = cohort_lookup[stay_id]
    features = process_stay_tensor(stay_id, stay_data, cohort_row)
    
    if features is not None:
        X_list.append(features)
        y_list.append(cohort_row['label'])
        stay_ids_processed.append(stay_id)
        subject_ids_processed.append(cohort_row['subject_id'])

# Create arrays
X_final = np.array(X_list, dtype=np.float32)
y_final = np.array(y_list, dtype=np.float32)
stay_ids_processed = np.array(stay_ids_processed)
subject_ids_processed = np.array(subject_ids_processed)

# Clean
X_final = np.nan_to_num(X_final, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\n{'='*70}")
print("EXTRACTION COMPLETE!")
print(f"{'='*70}")
print(f"   X_final: {X_final.shape}")
print(f"   y_final: {y_final.shape}")
print(f"   Sepsis: {int(y_final.sum()):,} ({y_final.mean()*100:.1f}%)")
print(f"   Controls: {int(len(y_final) - y_final.sum()):,}")
print(f"{'='*70}")

In [ ]:
# ============================================================================
# CELL 10: SAVE DATA
# ============================================================================

print("Saving data...")

np.save('X_final.npy', X_final)
np.save('y_final.npy', y_final)
np.save('stay_ids_processed.npy', stay_ids_processed)
np.save('subject_ids_processed.npy', subject_ids_processed)
cohort_df.to_csv('cohort_info.csv', index=False)

print("\n✅ Saved:")
print("   - X_final.npy")
print("   - y_final.npy")
print("   - stay_ids_processed.npy")
print("   - subject_ids_processed.npy")
print("   - cohort_info.csv")

# Download
try:
    from google.colab import files
    files.download('X_final.npy')
    files.download('y_final.npy')
    files.download('cohort_info.csv')
    print("\n📥 Files downloaded!")
except:
    print("\n   Files saved locally.")

print(f"\n🎉 Ready for model training!")
print(f"   Total samples: {len(X_final):,}")